# Spatial Accessibility with Pysal Access

**Authors:** Alexander Michels

In this notebook, we use the Pysall `access` package to calculate spatial accessibility to hospital beds using multiple methods.

In [1]:
import access
import contextily as cx
import geopandas as gpd
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import statsfrom scipy import stats
import sys

SyntaxError: invalid syntax (2056955394.py, line 9)

In [ ]:
pop_id = "FIPS"
figsize=(16, 14)

In [ ]:
pop = gpd.read_file("../data/svi/SVI2018_US_tract.shp")
pop = pop[pop["STCNTY"] == "17031"]
pop[pop_id] = pop[pop_id].astype(str)
pop = pop.to_crs("EPSG:4326")
print(len(pop))
pop.head()

In [ ]:
# load the hospitals
hos = gpd.read_file("../data/hospitals/ContinentalHospitals.shp")
hos = hos[(hos["STATE"] == "IL") & (hos["COUNTY"] == "COOK")]
hos = hos[hos["BEDS"] >= 0]
hos = hos.to_crs("EPSG:4326")
print(len(hos), sum(hos["BEDS"]))
hos.head()

In [ ]:
hospitals = gpd.sjoin(pop[["FIPS", "geometry"]], hos[["BEDS", "geometry"]], how="inner", predicate="intersects")
hospitals = hospitals.groupby("FIPS").agg({"BEDS": 'sum', "geometry": 'last'})
print(len(hospitals), sum(hospitals["BEDS"]))
hospitals.head()

In [ ]:
ax = pop.plot(column="E_TOTPOP", legend=True, figsize=(8,8), legend_kwds={'label': "Population per Tract", 'pad': 0.01})
ax = hos.plot(ax=ax, marker="+", color="orange", legend=True)
cx.add_basemap(ax=ax, crs=pop.crs.to_string(), attribution_size='4')
ax.axis('off')
plt.tight_layout()
plt.savefig("../figs/PopAndHospitals.png", dpi=200)
plt.show()

In [ ]:
# get car travel times
car = pd.read_csv("../data/travel_times/17031-output-TRACT-CAR.csv")
car["origin"] = car["origin"].astype(str)
car["destination"] = car["destination"].astype(str)
car.head()

In [ ]:
# calculate the gravity-based access
A = access.Access(
    demand_df=pop,
    demand_index=pop_id,
    demand_value="E_TOTPOP",
    supply_df=hospitals,
    supply_index=True,
    supply_value="BEDS",
    cost_df=car,
    cost_origin="origin",
    cost_dest="destination",
    cost_name="minutes",
    neighbor_cost_df=car,
    neighbor_cost_origin="origin",
    neighbor_cost_dest="destination",
    neighbor_cost_name="minutes"
)

In [ ]:
gravity = access.weights.gravity(scale=60, alpha=-2, min_dist=1)
e2sfca_weights_30 = access.weights.step_fn({10 : 1, 20 : 0.68, 30 : 0.22})
e2sfca_weights_60 = access.weights.step_fn({20 : 1, 40 : 0.68, 60 : 0.22})
gaussian10 = access.weights.gaussian(10)
gaussian20 = access.weights.gaussian(20)

In [ ]:
A.weighted_catchment(name="gravity", weight_fn=gravity)

In [ ]:
A.two_stage_fca(name="2sfca_30", max_cost=30)
A.two_stage_fca(name="2sfca_60", max_cost=60)

In [ ]:
A.enhanced_two_stage_fca(name="e2sfca_30", weight_fn=e2sfca_weights_30, max_cost=30)
A.enhanced_two_stage_fca(name="e2sfca_60", weight_fn=e2sfca_weights_60, max_cost=60)

In [ ]:
A.enhanced_two_stage_fca(name="g2sfca10", weight_fn=gaussian10)
A.enhanced_two_stage_fca(name="g2sfca20", weight_fn=gaussian20)

In [ ]:
A.three_stage_fca(name="3sfca")

In [ ]:
A.raam(name="raam30", tau=30)
A.raam(name="raam60", tau=60)

In [ ]:
print(len(A.access_df))
A.access_df.head()

In [ ]:
access_result = A.access_df.merge(pop[["FIPS", "geometry"]], left_index=True, right_on="FIPS").set_index("FIPS")
access_result = gpd.GeoDataFrame(access_result)
access_result = access_result.drop(columns=["E_TOTPOP"])
print(len(access_result))
print(type(access_result))
access_result.head()

In [ ]:
supply_field = "BEDS"
result_cols = [
    "gravity",
    "2sfca_30", "2sfca_60", "e2sfca_30", 
    "e2sfca_60", "g2sfca10", "g2sfca20",
    "3sfca", "raam30", "raam60"
]
result_cols = [x + f"_{supply_field}" for x in result_cols]
result_title_mapper = [
    "Gravity",
    "2SFCA 30 mins",
    "2SFCA 60 mins",
    "E2SFCA 30 mins",
    "E2SFCA 60 mins",
    "Gaussian 2SFCA 10",
    "Gaussian 2SFCA 20",
    "3SFCA",
    "RAAM 30",
    "RAAM 60"
]

In [ ]:
result = A.access_df.reset_index()
result[pop_id] = result[pop_id].astype(str)
# replace zeroes with nan to help distinguish between low and no access
for result_col in result_cols:
    result[result_col] = result[result_col].replace(0, np.nan)
    if "fca" in result_col:
        result[result_col] = result[result_col] * 1000
result.head()

In [ ]:
geom_result = gpd.GeoDataFrame(pop[[pop_id, "geometry"]].merge(result, on=pop_id))
geom_result = geom_result.set_crs(pop.crs)
for i, result_col in enumerate(result_cols):
    fig, ax = plt.subplots(figsize=figsize)
    ax = geom_result.plot(ax=ax, column=result_col, figsize=figsize, legend=True, antialiased=False,
                                missing_kwds= dict(color = "grey",), legend_kwds={'pad' : 0.01})
    ax.set_title(result_title_mapper[i])
    ax.margins(0.01)
    fig.subplots_adjust(hspace=0.0, wspace=0.0)
    ax.set_axis_off()
    cx.add_basemap(ax, attribution_size=6, crs=geom_result.crs.to_string())
geom_result.head()

In [ ]:
for col in access_result.columns:
    vals = access_result[col].replace(np.nan, 0)
    if col != "geometry":
        res = stats.normaltest(vals)
        print(col, res.statistic, res.pvalue)

In [ ]:
corr = access_result.corr(method='kendall', numeric_only=True)
corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
# access_result.to_file("../data/accessibility/CookHospitals.gpkg", driver="GPKG")